# Loading 

In [ ]:
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp
import time
import os

from spagrn.regulatory_network import InferNetwork as irn

In [ ]:
adata_hindbrain = sc.read_h5ad("Process_Data/bin100_3D_region/combined_adata_Hindbrain.h5ad")
adata_midbrain = sc.read_h5ad("Process_Data/bin100_3D_region/combined_adata_midbrain.h5ad")
adata_pons = sc.read_h5ad("Process_Data/bin100_3D_region/combined_adata_PONS.h5ad")
adata_hindbrain,adata_midbrain,adata_pons

In [ ]:
combined_adata = ad.concat(
    [adata_hindbrain,adata_midbrain,adata_pons],
    label='sample_id',     # Creates a new .obs column 'sample_id'
    keys=['hindbrain','midbrain','PONS'],             # Uses values from the 'keys' list (filenames) to fill 'sample_id'
    index_unique='-',      # Makes obs_names unique with the key and '-' (e.g., "9_D03...-AAACCCAAGGAG")
    join='outer'           # Keeps all genes from all files (takes the union)
)
combined_adata

In [ ]:
combined_adata.write_h5ad("Process_Data/bin100_3D_region/combined_adata_MHB.h5ad",compression='gzip')

# GRN inference

In [ ]:
tfs_fn = 'GRN_resource/hs_hgnc_tfs.txt'
database_fn = 'GRN_resource/hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather'
motif_anno_fn = 'GRN_resource/motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl'

niche_human = pd.read_csv('GRN_resource/lr_network_human.csv')
niche_mouse = pd.read_csv('GRN_resource/lr_network_mouse.csv')
niches = pd.concat([niche_mouse, niche_human])
print(niches)

In [ ]:
base_h5ad_path = "Process_Data/bin100_3D_region"
base_output_dir = "Output/MHB"

# The list of regions you provided
region_list = [
    "MHB",
]

print(f"--- GRN Inference Batch Processing Start ---")
print(f"Base H5AD path: {base_h5ad_path}")
print(f"Base output path: {base_output_dir}")

start_time_total = time.time()

# --- 2. Loop through each region ---
for region_name in region_list:
    print(f"\n{'='*40}")
    print(f"Processing Region: {region_name}")
    print(f"{'='*40}\n")
    
    start_time_region = time.time()

    # --- 2.1 Define dynamic paths ---
    h5ad_file = os.path.join(base_h5ad_path, f"combined_adata_{region_name}.h5ad")
    
    # Replace spaces in region name with underscores for a valid directory name
    safe_region_name = region_name.replace(' ', '_')
    output_dir_region = os.path.join(base_output_dir, safe_region_name)
    
    # Ensure the output directory exists
    os.makedirs(output_dir_region, exist_ok=True)
    
    print(f"  H5AD file: {h5ad_file}")
    print(f"  Output directory: {output_dir_region}")

    try:
        # --- 2.2 Load Data ---
        print(f"  [{region_name}] 1/6: Loading data...")
        if not os.path.exists(h5ad_file):
            raise FileNotFoundError(f"H5AD file not found: {h5ad_file}")
            
        adata_orig = sc.read_h5ad(h5ad_file)

        # --- 2.3 Preprocess ---
        print(f"  [{region_name}] 2/6: Preprocessing (min_cells={int(len(adata_orig.obs_names)*0.01)})...")
        # Filter using irn.preprocess
        adata = irn.preprocess(
            adata_orig, 
            min_genes=10, 
            min_cells=int(len(adata_orig.obs_names) * 0.01), 
            min_counts=10, 
            max_gene_num=20000
        )
        
        # Free original data memory
        del adata_orig 

        # --- 2.4 Prepare 'count' layer ---
        print(f"  [{region_name}] 3/6: Preparing 'count' layer...")
        X = adata.X
        if sp.isspmatrix_csc(X):
            adata.layers['count'] = X.copy()
        else:
            if sp.issparse(X):
                adata.layers['count'] = X.tocsc().copy()
            else:
                # dense numpy array -> convert to CSC sparse matrix
                adata.layers['count'] = sp.csc_matrix(X)
        del X
        print(f"  [{region_name}] AnnData after preprocessing: {adata}")

        # --- 2.5 Initialize GRN ---
        print(f"  [{region_name}] 4/6: Initializing GRN object...")
        grn = irn(adata, project_name=f'grn_{safe_region_name}')
        grn.add_params({'prune_auc_threshold': 0.05, 'rank_threshold': 9000, 'auc_threshold': 0.05})

        # --- 2.6 Infer GRN ---
        print(f"  [{region_name}] 5/6: Inferring GRN... (This may take a long time)")
        
        # Check if necessary variables are defined
        if 'database_fn' not in locals() or 'motif_anno_fn' not in locals() or 'tfs_fn' not in locals():
            raise NameError("Error: database_fn, motif_anno_fn, or tfs_fn are not defined. Please define them before the loop.")
            
        grn.infer(
            database_fn,
            motif_anno_fn,
            tfs_fn,
            niche_df=niches,
            gene_list=None,
            num_workers=15,
            cache=True, # Use cache, if this region has been run before
            output_dir=output_dir_region, # *** Key change: Use region-specific output directory ***
            save_tmp=True,
            layers='count',
            latent_obsm_key='align_spatial_3d', # Use 'align_spatial_3d' from reference code
            model='danb',
            n_neighbors=10,
            methods=['FDR_I','FDR_C','FDR_G'],
            operation='intersection',
            mode='geary',
            cluster_label='region_h2',
            torch_device='cuda:5',
            pair_batch_size=256,
            bivariate_pair_batch_size=96,
        )
        
        elapsed_region = time.time() - start_time_region
        print(f"  [{region_name}] 6/6: GRN inference complete.")
        print(f"--- Successfully completed Region: {region_name} (Duration: {time.strftime('%H:%M:%S', time.gmtime(elapsed_region))}) ---")

    except FileNotFoundError as e:
        print(f"[Error] {e}")
        print(f"Skipping Region: {region_name}")
    except NameError as e:
        print(f"[Critical Error] {e}")
        print("Please define the necessary database paths at the top of the script. Stopping execution.")
        break # Stop the entire loop if database paths are not defined
    except Exception as e:
        print(f"[!!! CRITICAL ERROR !!!] Failed to process Region '{region_name}':")
        print(f"  Error Type: {type(e)}")
        print(f"  Error Message: {e}")
        print(f"SkiTDF-8(Skipping Region: {region_name}")

elapsed_total = time.time() - start_time_total
print("\n===============================")
print(f"All specified regions have been processed.")
print(f"Total time elapsed: {time.strftime('%H:%M:%S', time.gmtime(elapsed_total))}")

